### Executar este notebook no Jupyter através do navegador

**URL:** *localhost:8888*  
**Senha:** *1234*

In [1]:
!pip install dotenv boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 71.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.7 MB/s eta 0:00:00


In [12]:
import os
import sys
from dotenv import load_dotenv

import boto3
from botocore.exceptions import ClientError

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, from_unixtime
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, BooleanType, LongType

load_dotenv()

True

In [13]:
def ensure_bucket_exists(bucket_name, region):
    s3_client = boto3.client(
        's3',
        aws_access_key_id=os.getenv('AWS_ACCESS_KEY'),
        aws_secret_access_key=os.getenv('AWS_SECRET_KEY'),
        region_name=region
    )

    try:
        s3_client.head_bucket(Bucket=bucket_name)
        print(f'Bucket {bucket_name} already exists.')
    except ClientError as e:
        error_code = e.response['Error']['Code']

        if error_code == '404':
            print(f'Bucket {bucket_name} not found. Creating...')
            location = {'LocationConstraint': region}
            s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration=location)
            print(f'Bucket {bucket_name} created')
        else:
            print(f"Can't verify bucket! {e}")

In [25]:
packages = [
    'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0',
    'org.apache.hadoop:hadoop-aws:3.3.4',
    'com.amazonaws:aws-java-sdk-bundle:1.12.262'
]

spark = SparkSession.builder \
    .appName("BinanceStreaming") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", ','.join(packages)) \
    .config("spark.hadoop.fs.s3a.access.key", os.getenv('AWS_ACCESS_KEY')) \
    .config("spark.hadoop.fs.s3a.secret.key", os.getenv('AWS_SECRET_KEY')) \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

In [27]:
raw_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'kafka-1:9092') \
    .option('subscribe', 'trades.normalized') \
    .option('startingOffsets', 'latest') \
    .load()

In [28]:
schema = StructType([
    StructField('event', StringType(), True),
    StructField('symbol', StringType(), True),
    StructField('trade_id', StringType(), True),
    StructField('price', DecimalType(18, 8), True),
    StructField('qty', DecimalType(18, 8), True),
    StructField('trade_time', LongType(), True),
    StructField('is_maker', BooleanType(), True)
])

In [29]:
df = raw_df.select(from_json(col('value').cast('string'), schema).alias('data'))
df = df.select('data.*')

In [30]:
df = df.withColumn(
    'trade_time_readable', 
    (from_unixtime(col('trade_time') / 1000)).cast('timestamp')
)

In [31]:
df = df.dropna()

In [32]:
# bucket_name = 'binance-data-lucas-2026'
# bucket_region = 'sa-east-1'

# ensure_bucket_exists(bucket_name, bucket_region)

# query_s3 = df.writeStream \
#     .format('parquet') \
#     .option('path', f's3a://{bucket_name}/data/normalized/') \
#     .option('checkpointLocation', f's3a://{bucket_name}/checkpoints/') \
#     .outputMode('append') \
#     .start()

# query_s3.awaitTermination()

In [33]:
query = df.writeStream \
    .queryName('binance_data') \
    .format('memory') \
    .outputMode('append') \
    .option('startingOffset', 'earliest') \
    .start()

In [ ]:
from IPython.display import clear_output
from time import sleep

while True:
    clear_output(wait=True)
    spark.sql('SELECT * FROM binance_data ORDER BY trade_time DESC LIMIT 10').show(truncate=False)
    sleep(5)

+-----+------+--------+-----+---+----------+--------+-------------------+
|event|symbol|trade_id|price|qty|trade_time|is_maker|trade_time_readable|
+-----+------+--------+-----+---+----------+--------+-------------------+
+-----+------+--------+-----+---+----------+--------+-------------------+



In [23]:
spark.stop()